# Bank QOS applicability audit (step 1) and cross-dataset summary

**What this is:** a *calculated / theoretical accounting* study using the same Theorem F.13 parameters as notebook 06 (Zhao et al., arXiv:2604.07639v2): sparsity s (row **or** column), κ_reg, test margins γ, and the F61 exponent decomposition. It is not a quantum simulation, not a resource estimate, and not evidence of quantum advantage.

**Known leakage caveat.** The audited matrix is notebook 05's *validated global encoding*: OneHotEncoder and StandardScaler are fitted on all 45,211 rows before CV (cell 6). Each fold's validation rows therefore influenced the encoding. This is the documented AGENTS.md limitation and is not corrected here.

**Conventions:**
- Unweighted ridge, no intercept, ‖X‖ = 1, λ = α/σ_max², and the same α grid as CFPB.
- Test sets are the validation rows of each of the 5 stratified CV folds (`make_stratified_cv`, seed 42), and yes = +1.
- D < N here, so σ_min comes from a dense SVD of each fold's training matrix with numpy's `matrix_rank` tolerance, and σ_min = 0 when rank < D (Eq. F50). The CFPB D > N path is unchanged.
- Bank's validated accounting convention is row-only s. Both conventions are reported side by side.
- The audit ridge is not notebook 05's RidgeClassifier, so its scores are not the regression-fixture scores.

**PaySim** is not onboarded. Its summary row is *analytic, not measured*.

In [1]:
# Run from the repository root, notebooks/, or another repository subdirectory.
from pathlib import Path
import sys, json

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "src/qos_audit.py").is_file() and (p / "AGENTS.md").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start this notebook with a working directory inside qos-rbc-experiments.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from IPython.display import display

from src import qos_audit
from src.cfpb_prepare import git_commit

## Configuration

In [2]:
OUTPUT_DIR = ROOT / "results/audit/bank_step1"
CFPB_DIR = ROOT / "results/audit/cfpb_step1"
SUMMARY_PATH = ROOT / "results/audit/summary_step1.csv"
CONFIG = dict(
    alphas=(0.1, 1.0, 10.0, 100.0),            # same grid as CFPB step 1
    delta=0.1,
    thresholds=(0.3, 0.1, 0.03, 0.01, 0.003),
    max_df_fractions=(1.0, 0.5, 0.2, 0.1, 0.05, 0.01),
    zipf_range=(10, 1000),                      # recorded for parity; not meaningful for D = 51
    seed=42,
    n_splits=5, cv_seed=42,                     # Bank's validated CV protocol
)
PAYSIM_ROWS = 6_362_620  # data rows in data/paysim.csv (6,362,621 lines incl. header); not onboarded

## Run the Bank audit (once)

The outputs are immutable. A completed run is only reloaded and its hashes verified.

In [3]:
manifest_path = OUTPUT_DIR / "manifest.json"
if manifest_path.exists():
    manifest = json.loads(manifest_path.read_text())
    if manifest.get("status") != "completed":
        raise RuntimeError(f"Incomplete audit run in {OUTPUT_DIR}; inspect it, do not overwrite")
    print("Reusing completed run", manifest["run_id"])
else:
    qos_audit.run_bank_step1(OUTPUT_DIR, **CONFIG)
    manifest = json.loads(manifest_path.read_text())
    print("Completed run", manifest["run_id"])
for name, digest in manifest["artifacts_sha256"].items():
    assert qos_audit._sha256(OUTPUT_DIR / name) == digest, f"Artifact drift: {name}"
assert manifest["config"]["alphas"] == list(CONFIG["alphas"])
folds = {t: json.loads((OUTPUT_DIR / f"{t}.json").read_text()) for t in manifest["tasks"]}
summary = pd.read_csv(OUTPUT_DIR / "summary.csv")
for caveat in manifest["caveats"]:
    print("CAVEAT:", caveat)

Reusing completed run 20261016T175910Z-a543a93d
CAVEAT: Known leakage: the validated encoded matrix is notebook 05's global fit (OneHotEncoder + StandardScaler on all 45,211 rows before CV), so fold validation rows influenced the encoding (AGENTS.md).
CAVEAT: Bank's validated accounting convention is row-only sparsity; this audit reports row-only and row-or-column s side by side.
CAVEAT: Bank is a pipeline-validation dataset, not evidence of practical quantum advantage.
CAVEAT: Audit ridge (LSQR on CSR, no intercept, ||X|| = 1) is not notebook 05's RidgeClassifier; its scores are not the regression-fixture scores.


## Ledger: append-only

In [4]:
log_path = ROOT / "results/experiment_log.csv"
new = pd.read_csv(OUTPUT_DIR / "ledger_rows.csv", dtype=str, keep_default_na=False)
before = log_path.read_bytes()
existing_ids = set(pd.read_csv(log_path, dtype=str, keep_default_na=False).experiment_id)
header = pd.read_csv(log_path, nrows=0).columns.tolist()
assert new.columns.tolist() == header, "Ledger column mismatch"
if existing_ids & set(new.experiment_id):
    assert set(new.experiment_id) <= existing_ids, "Partial ledger append detected"
    print("Ledger already contains", len(new), "rows for run", manifest["run_id"])
else:
    if not before.endswith(b"\n"):
        raise ValueError("Ledger does not end with a newline; refusing to append")
    new.to_csv(log_path, mode="a", header=False, index=False)
    assert log_path.read_bytes()[:len(before)] == before, "Existing ledger bytes changed"
    print("Appended", len(new), "ledger rows")

Ledger already contains 110 rows for run 20261016T175910Z-a543a93d


## Rank, σ_min and sparsity

In [5]:
svd = manifest["full_matrix_svd"]
print(f"Full matrix {manifest['representation']['N']} x {svd['D']}: rank {svd['rank']}, "
      f"sigma_max {svd['sigma_max']:.4f}, raw sigma_min {svd['sigma_min_raw']:.3e}, "
      f"smallest nonzero {svd['smallest_nonzero']:.4f}, tol {svd['tol']:.3e} -> sigma_min = {svd['sigma_min']}")
display(pd.DataFrame({t: dict(N_train=r["N_nonzero"], D=r["D_nonzero"], rank=r["svd"]["rank"],
                              sigma_max=r["sigma_max"], sigma_min_raw=r["svd"]["sigma_min_raw"],
                              smallest_nonzero=r["svd"]["smallest_nonzero"], sigma_min_used=r["sigma_min_scaled"],
                              s_row_only=r["sparsity"]["s_row_only"], s_row_or_col=r["sparsity"]["s_row_or_col"],
                              s_over_N=r["sparsity"]["s_row_or_col"] / r["N_nonzero"],
                              test_N=r["test_N"], test_s_sparse=r["test_sparsity"]["test_rows_s_sparse"],
                              all_lsqr_converged=all(a["lsqr"]["converged"] for a in r["alpha_sweep"]))
                      for t, r in folds.items()}).T)

Full matrix 45211 x 51: rank 43, sigma_max 455.6313, raw sigma_min 2.545e-13, smallest nonzero 15.0834, tol 4.574e-09 -> sigma_min = 0.0


,N_train,D,rank,sigma_max,sigma_min_raw,smallest_nonzero,sigma_min_used,s_row_only,s_row_or_col,s_over_N,test_N,test_s_sparse,all_lsqr_converged
cv_fold_1,36168,51,43,407.54094,0.0,13.816508,0.0,16,36168,1.0,9043,True,True
cv_fold_2,36169,51,43,407.531578,0.0,13.474551,0.0,16,36169,1.0,9042,True,True
cv_fold_3,36169,51,43,407.445503,0.0,13.674956,0.0,16,36169,1.0,9042,True,True
cv_fold_4,36169,51,43,407.513565,0.0,13.560446,0.0,16,36169,1.0,9042,True,True
cv_fold_5,36169,51,43,407.627979,0.0,12.896191,0.0,16,36169,1.0,9042,True,True


## Per-fold, per-α table

Cost columns are log10 polynomial prefactors, not resource estimates.

In [6]:
pd.set_option("display.width", 250, "display.max_columns", 40)
display(summary[["task_id", "alpha", "kappa_reg", "residual_ratio", "test_accuracy", "test_balanced_accuracy",
                 "gamma_min", "gamma_p10", "gamma_p50", "coverage@0.03", "A9_qubits",
                 "log10_F61_passes_gamma_min", "log10_F61_passes_gamma_p10"]].round(4))

,task_id,alpha,kappa_reg,residual_ratio,test_accuracy,test_balanced_accuracy,gamma_min,gamma_p10,gamma_p50,coverage@0.03,A9_qubits,log10_F61_passes_gamma_min,log10_F61_passes_gamma_p10
0,cv_fold_1,0.1,1288.7580,0.2895,0.9003,0.6315,0.0000,0.1297,0.5183,0.9736,52,67.2455,51.1247
1,cv_fold_1,1.0,407.5422,0.2896,0.9003,0.6315,0.0000,0.1299,0.5188,0.9736,52,64.6041,49.1217
2,cv_fold_1,10.0,128.8796,0.2899,0.9004,0.6308,0.0001,0.1320,0.5237,0.9739,52,60.1708,47.0940
3,cv_fold_1,100.0,40.7664,0.2930,0.8997,0.6226,0.0001,0.1454,0.5673,0.9776,52,57.7808,44.9265
4,cv_fold_2,0.1,1288.7284,0.2878,0.8984,0.6314,0.0000,0.1349,0.5096,0.9758,52,65.9118,51.0563
5,cv_fold_2,1.0,407.5328,0.2878,0.8985,0.6314,0.0002,0.1351,0.5101,0.9758,52,60.3061,49.0542
6,cv_fold_2,10.0,128.8767,0.2881,0.8981,0.6296,0.0001,0.1361,0.5148,0.9756,52,59.7783,47.0414
7,cv_fold_2,100.0,40.7654,0.2912,0.8984,0.6244,0.0000,0.1484,0.5554,0.9782,52,60.1015,44.8914
8,cv_fold_3,0.1,1288.4562,0.2875,0.8997,0.6292,0.0001,0.1310,0.5076,0.9770,52,63.1757,51.1063
9,cv_fold_3,1.0,407.4467,0.2875,0.8997,0.6292,0.0001,0.1311,0.5081,0.9771,52,61.3642,49.1056


## F61 exponent decomposition (γ = p10)

This is pure arithmetic on the saved fold JSON, and it must reproduce the reported F61.

In [7]:
DERIVED = OUTPUT_DIR / "derived" / "f61_decomposition_gamma_p10.csv"
decomposition = qos_audit.derive_f61_decomposition(OUTPUT_DIR, gamma_label="p10")
gap = (decomposition.set_index(["task_id", "alpha"])["log10_total"]
       - summary.set_index(["task_id", "alpha"])["log10_F61_passes_gamma_p10"]).abs().max()
assert gap < 1e-9, gap
if DERIVED.exists():
    pd.testing.assert_frame_equal(pd.read_csv(DERIVED), decomposition, check_exact=False, rtol=0, atol=1e-12)
    print("Derived table verified")
else:
    DERIVED.parent.mkdir(exist_ok=True)
    decomposition.to_csv(DERIVED, index=False)
    print("Wrote", DERIVED.relative_to(ROOT))
factor_cols = [f"log10_{f}" for f in qos_audit.F61_FACTORS]
display(decomposition[["task_id", "alpha", "s", "m", "min_term_branch", *factor_cols,
                       "log10_total", "log10_total_s1"]].round(2))
qos_audit.write_derived_manifest(OUTPUT_DIR, [str(DERIVED.relative_to(OUTPUT_DIR))])

Derived table verified


,task_id,alpha,s,m,min_term_branch,log10_s^7,log10_kappa^4,log10_gamma^-4,log10_delta^-1,log10_min_term,log10_total,log10_total_s1
0,cv_fold_1,0.1,36168,8138,log2^2 m,31.91,12.44,3.55,1.0,2.23,51.12,18.50
1,cv_fold_1,1.0,36168,8138,log2^2 m,31.91,10.44,3.55,1.0,2.23,49.12,16.49
2,cv_fold_1,10.0,36168,8138,log2^2 m,31.91,8.44,3.52,1.0,2.23,47.09,14.47
3,cv_fold_1,100.0,36168,8138,log2^2 m,31.91,6.44,3.35,1.0,2.23,44.93,12.30
4,cv_fold_2,0.1,36169,8137,log2^2 m,31.91,12.44,3.48,1.0,2.23,51.06,18.43
5,cv_fold_2,1.0,36169,8137,log2^2 m,31.91,10.44,3.48,1.0,2.23,49.05,16.43
6,cv_fold_2,10.0,36169,8137,log2^2 m,31.91,8.44,3.47,1.0,2.23,47.04,14.41
7,cv_fold_2,100.0,36169,8137,log2^2 m,31.91,6.44,3.31,1.0,2.23,44.89,12.26
8,cv_fold_3,0.1,36169,8137,log2^2 m,31.91,12.44,3.53,1.0,2.23,51.11,18.48
9,cv_fold_3,1.0,36169,8137,log2^2 m,31.91,10.44,3.53,1.0,2.23,49.11,16.48


{'artifacts_sha256': {'derived/f61_decomposition_gamma_p10.csv': '45b5ee1b9f1aef50048bba50d130463a43d1a3689f2d973ec97bddec77554d89'},
 'interpretation': 'calculated / theoretical accounting',
 'note': 'Post-run figures and derived tables; pure functions of the task JSONs',
 'source_manifest_sha256': 'ac7fe1f097a80bcdc51485ccfa908e7d23d4f6264834e29b605603b635fbcdc8',
 'source_run_id': '20261016T175910Z-a543a93d'}

## Cross-dataset summary

There is one row per dataset and task family, and ranges run across the tasks in each family.

- **Headline:** log10 F61 at γ = p10 at the α selected by validation accuracy. CFPB selects per task on the validation split. Bank selects one α by mean CV-fold accuracy (fold validation rows). Ties go to the larger α.
- **Secondary:** the cost-minimizing α. It is a grid-edge value (α = 100 everywhere), not an optimum.

The PaySim row is **analytic, not measured**, and its representation is not onboarded in this repo.

In [8]:
cfpb_summary = pd.read_csv(CFPB_DIR / "summary.csv")
cfpb_decomposition = pd.read_csv(CFPB_DIR / "derived" / "f61_decomposition_gamma_p10.csv")
def s7_range(decomposition):  # s does not depend on alpha
    return {"log10_s^7_min": float(decomposition["log10_s^7"].min()),
            "log10_s^7_max": float(decomposition["log10_s^7"].max())}
rows = []
for kind, family in (("pair", "CFPB pairwise (6)"), ("one_vs_rest", "CFPB one-vs-rest (4)")):
    ids = set(cfpb_summary.task_id[cfpb_summary.kind == kind])
    d = cfpb_decomposition[cfpb_decomposition.task_id.isin(ids)]
    rows.append(dict(qos_audit.family_summary(cfpb_summary[cfpb_summary.task_id.isin(ids)], d,
                                               dataset="CFPB Jan-Feb 2026 primary4", family=family,
                                               status="measured", select_by="val_accuracy"), **s7_range(d)))
rows.append(dict(qos_audit.family_summary(summary, decomposition, dataset="Bank Marketing (bank-full)",
                                           family="5 stratified CV folds (seed 42); global encoding, known leakage",
                                           status="measured", select_by="test_accuracy", pooled=True),
                **s7_range(decomposition)))
paysim = qos_audit.analytic_dense_column_row("PaySim", PAYSIM_ROWS)
paysim["log10_s^7_min"] = paysim["log10_s^7_max"] = paysim.pop("log10_s^7")
paysim.update(N_min=paysim.pop("N"), N_max=PAYSIM_ROWS, s_row_or_col_min=paysim["s_row_or_col"],
              s_row_or_col_max=paysim.pop("s_row_or_col"), s_over_N_min=1.0, s_over_N_max=paysim.pop("s_over_N"))
rows.append(paysim)
step1 = pd.DataFrame(rows)
for col in ("selected_alphas", "costmin_alphas_grid_edge"):
    step1[col] = step1[col].map(lambda a: "" if not isinstance(a, list) else ";".join(f"{x:g}" for x in a))
step1["n_tasks"] = step1["n_tasks"].astype("Int64")
if SUMMARY_PATH.exists():
    pd.testing.assert_frame_equal(pd.read_csv(SUMMARY_PATH, keep_default_na=True).astype(str),
                                  pd.read_csv(pd.io.common.StringIO(step1.to_csv(index=False))).astype(str))
    print("Cross-dataset summary verified:", SUMMARY_PATH.relative_to(ROOT))
else:
    step1.to_csv(SUMMARY_PATH, index=False)
    print("Wrote", SUMMARY_PATH.relative_to(ROOT))
display(step1.T)

Cross-dataset summary verified: results/audit/summary_step1.csv


,0,1,2,3
dataset,CFPB Jan-Feb 2026 primary4,CFPB Jan-Feb 2026 primary4,Bank Marketing (bank-full),PaySim
family,CFPB pairwise (6),CFPB one-vs-rest (4),5 stratified CV folds (seed 42); global encodi...,all (any dense column)
status,measured,measured,measured,"analytic, not measured; representation not onb..."
n_tasks,6,4,5,<NA>
selection,"per task, max val_accuracy; ties -> larger alpha","per task, max val_accuracy; ties -> larger alpha","one alpha for all tasks, max mean test_accurac...",NaN
selected_alphas,1;10,1,100,
log10_F61_p10_selected_alpha_min,36.683485,40.898818,44.817238,NaN
log10_F61_p10_selected_alpha_max,40.695304,42.874084,44.927534,NaN
log10_s1_floor_selected_alpha_min,11.058734,13.295742,12.189183,NaN
log10_s1_floor_selected_alpha_max,14.697931,15.271008,12.299479,NaN


## Derived-table ledger rows

There is one row each for the Bank decomposition table and the cross-dataset summary, matching the CFPB derived-table row. Each is appended only if absent.

In [9]:
def derived_ledger_row(row_id, dataset, artifact, metric, result, interpretation, **extra):
    notes = dict(interpretation=interpretation, artifact=str(artifact.relative_to(ROOT)),
                 artifact_sha256=qos_audit._sha256(artifact),
                 qos_audit_sha256=qos_audit._sha256(ROOT / "src/qos_audit.py"),
                 code_provenance="git_commit plus qos_audit_sha256 (derivation code may be uncommitted)", **extra)
    return dict(experiment_id=row_id, dataset=dataset, date=pd.Timestamp.now(tz="UTC").date().isoformat(),
                git_commit=git_commit(ROOT) or "", preprocessing="derived from saved step-1 audit outputs; no re-solve",
                n_samples="", n_features="", max_sparsity="", sketch_method="none", sketch_dimension="", seed=42,
                model="unweighted ridge, no intercept, ||X||=1 (step-1 fits; not refit)",
                metric=metric, result=result, notes=json.dumps(notes, sort_keys=True))
derived_rows = [
    derived_ledger_row(f"bank-qos-audit-step1-{manifest['run_id']}-f61-decomposition-gamma-p10",
                       "Bank Marketing (bank-full):5 CV folds", DERIVED, "f61_decomposition_table_rows",
                       len(decomposition),
                       "calculated / theoretical accounting; derived arithmetic on step-1 fold JSON, no re-solve",
                       derived_manifest=str((OUTPUT_DIR / "derived_manifest.json").relative_to(ROOT)),
                       source_run_id=manifest["run_id"], factors=list(qos_audit.F61_FACTORS),
                       counterfactual="s = 1, other measured factors kept",
                       caveat="global preprocessing before CV (known leakage)"),
    derived_ledger_row("qos-audit-step1-summary", "CFPB primary4 + Bank + PaySim (analytic)", SUMMARY_PATH,
                       "summary_step1_rows", len(step1),
                       "calculated / theoretical accounting; cross-dataset summary of step-1 outputs. "
                       "PaySim row analytic, not measured; representation not onboarded",
                       sources=[str((CFPB_DIR / "summary.csv").relative_to(ROOT)),
                                str((CFPB_DIR / "derived/f61_decomposition_gamma_p10.csv").relative_to(ROOT)),
                                str((OUTPUT_DIR / "summary.csv").relative_to(ROOT)), str(DERIVED.relative_to(ROOT))],
                       headline="log10 F61 at gamma p10 at validation-accuracy-selected alpha",
                       secondary="cost-minimizing alpha (grid edge)", summary_doc="docs/audit_step1_summary.md"),
]
existing_ids = set(pd.read_csv(log_path, dtype=str, keep_default_na=False).experiment_id)
for row in derived_rows:
    if row["experiment_id"] in existing_ids:
        print("Ledger already contains", row["experiment_id"]); continue
    before = log_path.read_bytes()
    pd.DataFrame([row])[header].to_csv(log_path, mode="a", header=False, index=False)
    assert log_path.read_bytes()[:len(before)] == before, "Existing ledger bytes changed"
    print("Appended", row["experiment_id"])

Ledger already contains bank-qos-audit-step1-20261016T175910Z-a543a93d-f61-decomposition-gamma-p10
Ledger already contains qos-audit-step1-summary


## Interpretation

All of the following was measured on Bank's validated global encoding, which carries the known leakage caveat above.

- **Rank and σ_min.** The full matrix and every fold's training matrix have rank 43 < D = 51. The 9 one-hot groups without `drop` give 8 linear dependencies. Raw σ_min is 2.5×10⁻¹³ against a tolerance of 4.6×10⁻⁹, so σ_min = 0 (Eq. F50), exactly as for CFPB. The smallest nonzero singular value is 0.033·σ_max.
- **Sparsity.** s_row_only = 16, matching the validated fixture. s_row_or_col = N_train (36,168–36,169), because the seven standardized numeric columns are dense.
- **κ_reg.** It is 41–1,289 across α ∈ {100, …, 0.1}. With σ_max ≈ 408, λ = α/σ_max² is small, and with σ_min = 0, κ_reg = √((1+λ)/λ) is set by λ alone.
- **Margins.** As-is test entries reach about 119 (standardized balance), so ‖x'‖_∞ ≤ 1 holds only after rescaling. Rescaled γ p10 is 0.13–0.15 and coverage at γ ≥ 0.03 is 97–98%.
- **Accuracy.** The audit ridge's test accuracy is 0.898–0.901 and balanced accuracy 0.621–0.638. These are close to notebook 05's paper-style Ridge (0.8995 / 0.630), but this is a different model (no intercept, LSQR, ‖X‖ = 1), not the regression fixture.
- **F61 at γ = p10.** It is 10^44.8–10^51.1 and is lowest at α = 100, the grid edge. Mean CV accuracy selects α = 100 as well, but only by about 1×10⁻⁴ (an effective tie), and balanced accuracy peaks at α = 0.1. s⁷ supplies 31.9 decades and κ_reg⁴ supplies 6.4–12.4. The s = 1 floor is still 10^12.2–10^18.5.

The cross-dataset summary (`results/audit/summary_step1.csv`, `docs/audit_step1_summary.md`) shows that the sparsity condition fails on every dataset. Bank additionally has a large κ_reg. The PaySim row is analytic, not measured.
